# Hybrid RAG Search System — Kaggle Ready

This notebook is a cleaned, dependency-safe version of the Hybrid RAG project.

### Pipeline
PDF → page extraction → recursive chunking → embeddings → FAISS semantic search → cosine similarity → TF-IDF keyword search → RRF hybrid search → cross-encoder reranking → FLAN-T5 generation → faithfulness proxy → Precision@K / Recall@K

**Why this version:** it does not depend on LangChain/LangGraph, so the `langgraph` / `langchain-core` dependency conflict from the original notebook is avoided.

> Run the cells from top to bottom. If Kaggle asks to restart the session after installation, restart once and continue from the import cell.

## 1. Install Kaggle Dependencies

Only the packages needed by this notebook are installed. LangChain and LangGraph are intentionally not used.

In [ ]:
# Kaggle package installation
# Run this cell once.

!pip install -q pypdf sentence-transformers faiss-cpu scikit-learn pandas numpy transformers sentencepiece

print("Required packages installed.")

## 2. Import Libraries

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import faiss
import torch

print("All imports successful!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 3. Locate the Uploaded PDF

The notebook automatically searches `/kaggle/input`.

It prefers the **Beyond Possible - Nims Purja** PDF used in the original project.

In [ ]:
# Find PDF files in Kaggle input

pdf_files = list(Path("/kaggle/input").rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        "No PDF found. In Kaggle, click Add Data and attach your PDF."
    )

print("PDF files found:")
for i, path in enumerate(pdf_files):
    print(i, ":", path)

preferred = [
    p for p in pdf_files
    if "Beyond Possible" in p.name
]

pdf_file = preferred[0] if preferred else pdf_files[0]

print("\nUsing PDF:")
print(pdf_file)

## 4. Extract PDF Pages

In [ ]:
reader = PdfReader(str(pdf_file))

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    text = text.strip()

    if text:
        pages.append({
            "page_number": page_number,
            "text": text
        })

print("Number of non-empty pages:", len(pages))

if not pages:
    raise ValueError("No text could be extracted from the PDF.")

print("\nFirst page preview:\n")
print(pages[0]["text"][:1500])

## 5. Recursive Text Chunking

Chunk size: **400 characters**

Overlap: **50 characters**

In [ ]:
def recursive_split_text(text, chunk_size=400, chunk_overlap=50):
    if not text.strip():
        return []

    separators = ["\n\n", "\n", ". ", " ", ""]

    def split_recursive(text, separator_index=0):
        if len(text) <= chunk_size:
            return [text]

        separator = separators[separator_index]

        if separator == "":
            pieces = [
                text[i:i + chunk_size]
                for i in range(0, len(text), chunk_size)
            ]
        else:
            raw_parts = text.split(separator)
            pieces = []
            current = ""

            for part in raw_parts:
                candidate = (
                    part if not current
                    else current + separator + part
                )

                if len(candidate) <= chunk_size:
                    current = candidate
                else:
                    if current.strip():
                        pieces.append(current.strip())
                    current = part

            if current.strip():
                pieces.append(current.strip())

        if separator_index < len(separators) - 1:
            final_pieces = []
            for piece in pieces:
                if len(piece) > chunk_size:
                    final_pieces.extend(
                        split_recursive(piece, separator_index + 1)
                    )
                else:
                    final_pieces.append(piece)
            pieces = final_pieces

        return pieces

    pieces = split_recursive(text)

    # Add overlap
    chunks = []
    for i, piece in enumerate(pieces):
        piece = piece.strip()
        if not piece:
            continue

        if i > 0 and chunk_overlap > 0:
            previous = pieces[i - 1]
            overlap_text = previous[-chunk_overlap:]
            piece = overlap_text + " " + piece

        chunks.append(piece[:chunk_size])

    return chunks


BASE_CHUNK_SIZE = 400
BASE_OVERLAP = 50

base_chunks = []

for page in pages:
    chunks = recursive_split_text(
        page["text"],
        chunk_size=BASE_CHUNK_SIZE,
        chunk_overlap=BASE_OVERLAP
    )

    for chunk in chunks:
        base_chunks.append({
            "chunk_id": len(base_chunks),
            "page_number": page["page_number"],
            "text": chunk
        })

print("Total chunks:", len(base_chunks))

for item in base_chunks[:3]:
    print("=" * 70)
    print("Chunk ID:", item["chunk_id"])
    print("Page:", item["page_number"])
    print(item["text"][:500])

## 6. Create Embedding Model

Model: `sentence-transformers/all-MiniLM-L6-v2`

Embeddings are normalized so inner-product search is equivalent to cosine similarity.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded.")
print("Device:", device)

## 7. Generate Document Embeddings

In [ ]:
document_texts = [
    item["text"]
    for item in base_chunks
]

document_embeddings = embedding_model.encode(
    document_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", document_embeddings.shape)

## 8. Create FAISS Vector Database

Because the embeddings are normalized, `IndexFlatIP` gives cosine-similarity-equivalent scores.

In [ ]:
dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(document_embeddings)

print("FAISS index created.")
print("Number of vectors:", faiss_index.ntotal)
print("Embedding dimension:", dimension)

## 9. Semantic Search + Cosine Similarity

In [ ]:
def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, len(base_chunks))
    )

    results = []

    for rank, (index, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):
        if index < 0:
            continue

        item = base_chunks[int(index)]

        results.append({
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "page_number": item["page_number"],
            "score": float(score),
            "document": item
        })

    return results


query = "What equipment is required for mountaineering?"

semantic_results = semantic_search(
    query,
    top_k=5
)

print("SEMANTIC SEARCH RESULTS\n")

for result in semantic_results:
    print("=" * 70)
    print("Rank:", result["rank"])
    print("Chunk ID:", result["chunk_id"])
    print("Page:", result["page_number"])
    print("Cosine Similarity:", round(result["score"], 4))
    print(result["document"]["text"][:600])

## 10. Cosine Similarity Before Hybrid Search

The top-1 / top-2 difference shows how much the best semantic result is separated from the second result.

In [ ]:
def show_cosine_similarity(query, top_k=5):
    results = semantic_search(query, top_k=top_k)

    df = pd.DataFrame([
        {
            "Rank": r["rank"],
            "Chunk ID": r["chunk_id"],
            "Page": r["page_number"],
            "Cosine Similarity": round(r["score"], 4)
        }
        for r in results
    ])

    display(df)

    if len(results) >= 2:
        difference = results[0]["score"] - results[1]["score"]
        print(
            "Top-1 minus Top-2 cosine difference:",
            round(float(difference), 4)
        )

    return df


cosine_before_hybrid_df = show_cosine_similarity(
    "What equipment is required for mountaineering?",
    top_k=5
)

## 11. TF-IDF Keyword Search

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    document_texts
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
def keyword_search(query, top_k=10):
    query_vector = tfidf_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = base_chunks[int(index)]

        results.append({
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "page_number": item["page_number"],
            "score": float(scores[index]),
            "document": item
        })

    return results


keyword_results = keyword_search(
    "mountaineering equipment",
    top_k=5
)

print("KEYWORD SEARCH RESULTS\n")

for result in keyword_results:
    print("=" * 70)
    print("Rank:", result["rank"])
    print("Chunk ID:", result["chunk_id"])
    print("TF-IDF Score:", round(result["score"], 4))
    print(result["document"]["text"][:500])

## 12. Hybrid Search — Reciprocal Rank Fusion

The hybrid retriever combines semantic and keyword rankings.

RRF:

`score = 1 / (60 + rank)`

In [ ]:
def hybrid_search(
    query,
    top_k=5,
    semantic_k=10,
    keyword_k=10,
    rrf_constant=60
):
    semantic_results = semantic_search(
        query,
        top_k=semantic_k
    )

    keyword_results = keyword_search(
        query,
        top_k=keyword_k
    )

    rrf_scores = {}
    document_map = {}

    for rank, result in enumerate(
        semantic_results,
        start=1
    ):
        chunk_id = result["chunk_id"]

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0.0)
            + 1.0 / (rrf_constant + rank)
        )

        document_map[chunk_id] = result["document"]

    for rank, result in enumerate(
        keyword_results,
        start=1
    ):
        chunk_id = result["chunk_id"]

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0.0)
            + 1.0 / (rrf_constant + rank)
        )

        document_map[chunk_id] = result["document"]

    ranked = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []

    for rank, (chunk_id, score) in enumerate(
        ranked,
        start=1
    ):
        results.append({
            "rank": rank,
            "chunk_id": chunk_id,
            "rrf_score": float(score),
            "document": document_map[chunk_id]
        })

    return results


hybrid_results = hybrid_search(
    "What equipment is required for mountaineering?",
    top_k=5
)

print("HYBRID SEARCH RESULTS\n")

for result in hybrid_results:
    print("=" * 70)
    print("Rank:", result["rank"])
    print("Chunk ID:", result["chunk_id"])
    print("RRF Score:", round(result["rrf_score"], 6))
    print(result["document"]["text"][:600])

## 13. Cross-Encoder Reranking

In [ ]:
cross_encoder_device = "cuda" if torch.cuda.is_available() else "cpu"

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=cross_encoder_device
)

print("Cross-encoder loaded.")
print("Device:", cross_encoder_device)

In [ ]:
def hybrid_rerank(
    query,
    candidate_k=10,
    top_k=3
):
    candidates = hybrid_search(
        query,
        top_k=candidate_k
    )

    pairs = [
        (
            query,
            item["document"]["text"]
        )
        for item in candidates
    ]

    scores = cross_encoder.predict(pairs)

    reranked = []

    for item, score in zip(candidates, scores):
        reranked.append({
            "chunk_id": item["chunk_id"],
            "page_number": item["document"]["page_number"],
            "rrf_score": item["rrf_score"],
            "cross_encoder_score": float(score),
            "document": item["document"]
        })

    reranked.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    for rank, item in enumerate(reranked[:top_k], start=1):
        item["rank"] = rank

    return reranked[:top_k]


reranked_results = hybrid_rerank(
    "What equipment is required for mountaineering?",
    candidate_k=10,
    top_k=5
)

print("RERANKED RESULTS\n")

for result in reranked_results:
    print("=" * 70)
    print("Rank:", result["rank"])
    print("Chunk ID:", result["chunk_id"])
    print(
        "Cross-Encoder Score:",
        round(result["cross_encoder_score"], 4)
    )
    print(result["document"]["text"][:600])

## 14. Load FLAN-T5 for Grounded Answer Generation

Use `text2text-generation` because FLAN-T5 is a sequence-to-sequence model.

In [ ]:
from transformers import pipeline

generation_device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=generation_device
)

print("FLAN-T5 loaded successfully.")

## 15. Hybrid RAG Answer Function

In [ ]:
def format_context(documents):
    parts = []

    for i, doc in enumerate(documents, start=1):
        parts.append(
            f"[Source {i} | Page {doc['page_number']} | Chunk {doc['chunk_id']}]\n"
            f"{doc['text']}"
        )

    return "\n\n".join(parts)


def hybrid_rag_answer(
    question,
    candidate_k=10,
    top_k=3
):
    start_time = time.time()

    reranked = hybrid_rerank(
        question,
        candidate_k=candidate_k,
        top_k=top_k
    )

    documents = [
        result["document"]
        for result in reranked
    ]

    context = format_context(documents)

    prompt = f"""Answer the question using ONLY the context below.

If the answer is not contained in the context, say:
I don't know based on the provided context.

Context:
{context}

Question:
{question}

Answer:"""

    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )

    answer = response[0]["generated_text"].strip()

    latency = time.time() - start_time

    return {
        "answer": answer,
        "documents": documents,
        "reranked": reranked,
        "latency": latency
    }

## 16. Test Complete RAG Pipeline

In [ ]:
question = "What equipment is required for mountaineering?"

result = hybrid_rag_answer(
    question,
    candidate_k=10,
    top_k=3
)

print("=" * 70)
print("QUESTION")
print("=" * 70)
print(question)

print("\n" + "=" * 70)
print("GENERATED ANSWER")
print("=" * 70)
print(result["answer"])

print("\nLatency:",
      round(result["latency"], 3),
      "seconds")

## 17. Display Retrieved Sources

In [ ]:
print("=" * 70)
print("RETRIEVED SOURCES")
print("=" * 70)

for i, doc in enumerate(
    result["documents"],
    start=1
):
    print(
        f"\nSource {i} "
        f"(Page {doc['page_number']}, "
        f"Chunk {doc['chunk_id']}):"
    )
    print(doc["text"][:500])

## 18. Faithfulness Proxy

This is an **embedding-based faithfulness proxy**. It measures semantic similarity between the generated answer and each retrieved source.

It is useful as a project metric, but it is **not a formal entailment or factuality guarantee**.

In [ ]:
def faithfulness_test(
    answer,
    documents,
    threshold=0.55
):
    if not answer.strip() or not documents:
        return {
            "faithfulness_score": 0.0,
            "status": "FAIL",
            "source_scores": []
        }

    answer_embedding = embedding_model.encode(
        [answer],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    source_embeddings = embedding_model.encode(
        [doc["text"] for doc in documents],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores = cosine_similarity(
        answer_embedding,
        source_embeddings
    )[0]

    best_score = float(np.max(scores))

    return {
        "faithfulness_score": round(best_score, 4),
        "status": (
            "PASS"
            if best_score >= threshold
            else "FAIL"
        ),
        "source_scores": [
            round(float(score), 4)
            for score in scores
        ]
    }


faithfulness_result = faithfulness_test(
    result["answer"],
    result["documents"]
)

print("=" * 70)
print("FAITHFULNESS TEST")
print("=" * 70)
print(
    "Faithfulness Score:",
    faithfulness_result["faithfulness_score"]
)
print(
    "Status:",
    faithfulness_result["status"]
)
print(
    "Source Scores:",
    faithfulness_result["source_scores"]
)

## 19. Precision@5 and Recall@5

These metrics need manually labelled relevant chunk IDs.

The first question below uses chunk IDs observed in the original notebook. For the other questions, inspect the retrieved results and add the relevant IDs before treating their metrics as meaningful.

In [ ]:
def precision_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    retrieved = retrieved_ids[:k]

    if not retrieved:
        return 0.0

    hits = sum(
        1
        for doc_id in retrieved
        if doc_id in relevant_ids
    )

    return hits / len(retrieved)


def recall_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    retrieved = retrieved_ids[:k]

    if not relevant_ids:
        return 0.0

    hits = sum(
        1
        for doc_id in retrieved
        if doc_id in relevant_ids
    )

    return hits / len(relevant_ids)

In [ ]:
evaluation_data = [
    {
        "question": "What equipment is required for mountaineering?",
        "relevant_ids": [248, 1057]
    },
    {
        "question": "What skills are important for climbing?",
        "relevant_ids": []
    },
    {
        "question": "How should climbers prepare for mountains?",
        "relevant_ids": []
    }
]

evaluation_results = []

for item in evaluation_data:
    start = time.time()

    results = hybrid_search(
        item["question"],
        top_k=5
    )

    latency = time.time() - start

    retrieved_ids = [
        r["chunk_id"]
        for r in results
    ]

    evaluation_results.append({
        "Question": item["question"],
        "Retrieved IDs": retrieved_ids,
        "Relevant IDs": item["relevant_ids"],
        "Precision@5": precision_at_k(
            retrieved_ids,
            item["relevant_ids"],
            5
        ),
        "Recall@5": recall_at_k(
            retrieved_ids,
            item["relevant_ids"],
            5
        ),
        "Latency (sec)": round(latency, 4)
    })

evaluation_df = pd.DataFrame(
    evaluation_results
)

display(evaluation_df)

## 20. Interactive Question Answering

Type `exit` to stop.

In [ ]:
while True:
    question = input(
        "\nAsk a question about the book "
        "(type 'exit' to stop): "
    )

    if question.lower().strip() == "exit":
        print("Program stopped.")
        break

    result = hybrid_rag_answer(
        question,
        candidate_k=10,
        top_k=3
    )

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(result["answer"])

    print("\n" + "=" * 70)
    print("SOURCES")
    print("=" * 70)

    for i, doc in enumerate(
        result["documents"],
        start=1
    ):
        print(
            f"\nSource {i} — "
            f"Page {doc['page_number']}, "
            f"Chunk {doc['chunk_id']}"
        )
        print(doc["text"][:300])

    print(
        "\nLatency:",
        round(result["latency"], 3),
        "seconds"
    )

# Optional: Streamlit Dashboard

The core RAG notebook above is the part you should get working first.

A Streamlit dashboard can be added after the notebook runs successfully. Kaggle may restrict public tunnelling, so the dashboard is kept separate from the core pipeline.

In [ ]:
# Optional package for the dashboard
!pip install -q streamlit

print("Streamlit installed.")

## Final Project Flow

**Input:** PDF

**Retrieval:**
- Semantic search
- Cosine similarity
- TF-IDF keyword search
- RRF hybrid search

**Ranking:**
- Cross-encoder reranking

**Generation:**
- FLAN-T5

**Evaluation:**
- Faithfulness proxy
- Precision@5
- Recall@5
- Latency

**Output:** Grounded answer + retrieved sources